In [ ]:
# Broken links
"""https://fair-chem.github.io/core/datasets/oc20.html 	HTTP 404 	Dataset Content
https://fair-chem.github.io/core/datasets/omat24.html 	HTTP 404 	Dataset Content
https://doi.org/10.5281/zenodo.5109599 	HTTP 502 	Dataset Content
https://doi.org/10.5281/zenodo.7905585 	HTTP 502 	Dataset Content
https://github.com/Open-Catalyst-Project/ocp/blob/main/DATASET.md#open-catalyst-2022-oc22 	HTTP 404 	Dataset Content
https://doi.org/10.1021/acs.organomet.2c00238.s001"""

In [ ]:
from colabfit.tools.vast.database import get_session
import pyarrow as pa

In [ ]:
link = "https://doi.org/10.1021/acs.organomet.2c00238/suppl_file/om2c00238_si_001.xyz"
new_link = "https://pubs.acs.org/doi/suppl/10.1021/acs.organomet.2c00238/suppl_file/om2c00238_si_001.xyz"
with get_session().transaction() as tx:
    table = tx.bucket("colabfit-prod").schema("prod").table("ds")
    reader = table.select(
        columns=["id", "name", "links"],
        predicate=table["links"].contains(f"'{link}'"),
        internal_row_id=True,
    )
    rows = reader.read_all()
    new_rows = []
    for row in rows.to_pylist():
        links = row["links"]
        new_rows.append(
            {
                "name": row["name"],
                "id": row["id"],
                "links": links.replace(link, new_link),
                "$row_id": row["$row_id"],
            }
        )
    new_table = pa.Table.from_pylist(new_rows, schema=rows.schema)
    table.update(new_table)